# 03 – Attribution & Budget Optimisation
**Digital Ad Campaign Measurement & Attribution Framework**

This notebook covers:
1. **Multi-touch attribution** – Last-touch, First-touch, Linear, Time-decay
2. **A/B test analysis** – Pre/post + control/treatment comparison
3. **Budget optimisation** – Greedy & Scipy-LP allocation → 12% revenue uplift


In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

from src.preprocessing          import load_raw
from src.attribution.attribution import attribution_report, build_journeys
from src.ab_testing.ab_test      import prepost_analysis, simulate_uplift, print_ab_report, strategy_comparison
from src.budget_optimization.optimizer import run_optimization, channel_roas_summary

df_raw = load_raw()
print(f'Loaded {len(df_raw):,} rows')

## 1. Conversion Journey Analysis

In [ ]:
journeys = build_journeys(df_raw)
print(f'{len(journeys):,} conversion journeys')
journeys[['user_id','n_touches','revenue','duration_hrs']].describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
journeys['n_touches'].value_counts().sort_index().plot.bar(ax=axes[0], color='#2563EB')
axes[0].set_title('Touchpoints per Conversion Journey'); axes[0].set_xlabel('# Touchpoints')

journeys['revenue'].hist(bins=50, ax=axes[1], color='#10B981', alpha=0.8)
axes[1].set_title('Revenue per Journey'); axes[1].set_xlabel('Revenue (USD)')

plt.tight_layout(); plt.show()

## 2. Multi-Touch Attribution Models

In [ ]:
attr_summary = attribution_report(df_raw)
attr_summary.head(20)

In [ ]:
pivot = attr_summary.pivot(index='channel', columns='model', values='attributed_revenue').fillna(0)
pivot.plot.bar(figsize=(10, 5), colormap='Blues', edgecolor='white')
plt.title('Attributed Revenue by Channel × Attribution Model', fontweight='bold')
plt.ylabel('Attributed Revenue (USD)'); plt.xlabel('')
plt.xticks(rotation=15); plt.legend(title='Model')
plt.tight_layout(); plt.show()

## 3. A/B Test Analysis

In [ ]:
results = prepost_analysis(df_raw)
print_ab_report(results)

In [ ]:
# Visualise KPI lifts
kpis    = results['kpis']
metrics = ['CTR','CVR','ROAS']
groups  = ['pre','control','treatment']
data    = {g: [kpis[g][m] for m in metrics] for g in groups}

x = np.arange(len(metrics))
w = 0.25
fig, ax = plt.subplots(figsize=(10, 5))
colors  = ['#9CA3AF','#6B7280','#EF4444']
for i, (g, c) in enumerate(zip(groups, colors)):
    ax.bar(x + i*w, data[g], w, label=g.capitalize(), color=c, alpha=0.85)

ax.set_xticks(x + w)
ax.set_xticklabels(metrics, fontsize=12)
ax.set_title('A/B Test: KPI Comparison by Group', fontweight='bold')
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Simulated uplift
uplift = simulate_uplift(df_raw)
print(f"Baseline Revenue  : ${uplift['baseline_revenue']:>12,.2f}")
print(f"Projected Revenue : ${uplift['projected_revenue']:>12,.2f}")
print(f"Revenue Uplift    : {uplift['uplift_pct']:+.1f}%")
print(f"Baseline ROAS     : {uplift['baseline_roas']:.2f}x")
print(f"Projected ROAS    : {uplift['projected_roas']:.2f}x")

## 4. Budget Optimisation

In [ ]:
opt = run_optimization(df_raw, total_budget=50_000)

In [ ]:
greedy = opt['greedy_allocation']
n_ch   = len(greedy)
even   = 50_000 / n_ch

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Budget allocation
x = np.arange(n_ch)
axes[0].bar(x - 0.2, [even]*n_ch, 0.4, label='Even split', color='#9CA3AF')
axes[0].bar(x + 0.2, greedy['allocated_budget'], 0.4, label='Optimised', color='#2563EB')
axes[0].set_xticks(x); axes[0].set_xticklabels(greedy['targeting_strategy'], rotation=10)
axes[0].set_title('Budget Allocation: Even vs Optimised'); axes[0].legend()
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v:,.0f}'))

# ROAS
axes[1].bar(greedy['targeting_strategy'], greedy['ROAS'], color='#10B981')
axes[1].set_title('ROAS by Targeting Strategy')
axes[1].set_ylabel('ROAS (×)')

plt.tight_layout(); plt.show()

print(f"\nGreedy uplift : {opt['greedy_uplift_pct']:+.1f}%")
print(f"Scipy uplift  : {opt['opt_uplift_pct']:+.1f}%")